Now we're going to check other possible candidates to see if they display transcriptomic significance or near-significance.

Bridges from notebook 9's R output

In [6]:
import os
os.chdir('/home/ethan-xiao/food-allergy-biomarkers')

print(os.getcwd())
os.listdir(".")

os.listdir("data")

/home/ethan-xiao/food-allergy-biomarkers


['within_cohort_null.npy',
 'raw_114134.csv',
 'grSet_funnorm.rds',
 'foldA_top2000_raw.csv',
 'sample_info.csv',
 'bumphunter_114134.rds',
 'perm_null_A.npy',
 'combat_diagnostics.rds',
 'raw_189148.csv',
 'combat_noMod.npy',
 'allergy_status.rds',
 'detP.rds',
 'combat_noMod.csv',
 'snp_pairwise.csv',
 'GSE189148_sample_metadata.csv',
 'within_cohort_null_L2.npy',
 'rgSet_resting.rds',
 'mVals_114134.rds',
 'probe_ids.csv',
 'GSE114134_family.soft.gz',
 'GSE34639.txt',
 'tt_189148.rds',
 'GSE189148_sample_metadata_snp_annotated.csv',
 'batch.rds',
 'scan_dates_189148.rds',
 'GSE189149_sample_metadata.csv',
 'GSE189148_resting_sample_metadata.csv',
 'GSE114135_family.soft.gz',
 'tt_114134.rds',
 'top20_candidates_for_rnaseq_check.csv',
 'GSE189149_raw_counts_GRCh38.p13_NCBI(1).tsv.gz',
 'GSE189148_suppl',
 'snp_pairwise.rds',
 'GSE189148.txt',
 'probes_to_keep.rds',
 'GSE114135.txt',
 'GSE189148_idats',
 'GSE114134.txt',
 'mVals_114134_shared.rds',
 'GSE114134_eset.rds',
 'foldB_top20

In [7]:
import pandas as pd

candidates = pd.read_csv("data/top20_candidates_for_rnaseq_check.csv")
rnaseq_results = pd.read_csv("data/deseq2_results_GSE189149.csv", index_col=0)

print(candidates.shape)
print(rnaseq_results.shape)
candidates.head()

(20, 13)
(24745, 6)


,chr,start_infant,end_infant,value_infant,p_infant,start_adol,end_adol,value_adol,p_adol,direction_matched,combined_p,rank,genes
0,chr12,739953,740338,0.685943,0.000016,739953,740338,0.493781,0.000246,True,8.183002e-08,1,NINJ2; LOC100049716;NINJ2;NINJ2
1,chr2,30669597,30669863,0.769272,0.000020,30669597,30669863,0.545559,0.000283,True,1.138820e-07,2,LCLAT1;LCLAT1; LCLAT1;LCLAT1;LCLAT1;LCLAT1
2,chr8,11666017,11666810,-0.331600,0.000156,11666281,11666594,-0.461489,0.000181,True,5.194028e-07,3,FDFT1; FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDF...
3,chr17,180404,181288,0.445629,0.000309,180404,181288,0.685564,0.000118,True,6.597698e-07,4,RPH3AL; RPH3AL;RPH3AL;RPH3AL;RPH3AL;LOC1005063...
4,chr17,33759512,33759573,-0.310298,0.004847,33759512,33760971,-0.530565,0.000009,True,7.786294e-07,5,SLFN12; SLFN12;SLFN12


In [8]:
def clean_genes(gene_str):
    parts = gene_str.replace(';', ' ').split()
    return sorted(set(parts))

candidates['gene_list'] = candidates['genes'].apply(clean_genes)

all_genes = sorted(set(g for genes in candidates['gene_list'] for g in genes))
print(len(all_genes))
print(all_genes[:20])

23
['ARRB1', 'C15orf26', 'C4orf37', 'CYP2E1', 'FDFT1', 'LCLAT1', 'LOC100049716', 'LOC100506388', 'LOC399886', 'MRI1', 'NINJ2', 'OR2L13', 'PRKCZ', 'RGS14', 'RPH3AL', 'SLFN12', 'STPG2', 'SUN1', 'UGT2B15', 'UGT2B17']


The RNA-seq results are indexed by Entrez ID, not gene symbol, so this maps each candidate region's gene symbol(s) to an Entrez ID before they can be joined.

In [10]:
import mygene

mg = mygene.MyGeneInfo()

loc_genes = [g for g in all_genes if g.startswith("LOC")]
named_genes = [g for g in all_genes if not g.startswith("LOC")]

loc_mapping = {g: g.replace("LOC", "") for g in loc_genes}

query_result = mg.querymany(named_genes, scopes='symbol', fields='entrezgene', species='human')
named_mapping = {r['query']: str(r.get('entrezgene', None)) for r in query_result}

gene_to_entrez = {**loc_mapping, **named_mapping}
print(gene_to_entrez)

4 input query terms found no hit:	['C15orf26', 'C4orf37', 'UNC84A', 'WDR66']


{'LOC100049716': '100049716', 'LOC100506388': '100506388', 'LOC399886': '399886', 'ARRB1': '408', 'C15orf26': 'None', 'C4orf37': 'None', 'CYP2E1': '1571', 'FDFT1': '2222', 'LCLAT1': '253558', 'MRI1': '84245', 'NINJ2': '4815', 'OR2L13': '284521', 'PRKCZ': '5590', 'RGS14': '10636', 'RPH3AL': '9501', 'SLFN12': '55106', 'STPG2': '285555', 'SUN1': '23353', 'UGT2B15': '7366', 'UGT2B17': '7367', 'UNC84A': 'None', 'WDR66': 'None', 'ZNF718': '255403'}


For the four that didn't work out: UNC84A is an old naming convention for SUN1, which is already here, same for WDR66 and CFAP251, and now will check the other two alongside WDR66

In [11]:
retry_result = mg.querymany(
    ['C15orf26', 'C4orf37', 'WDR66'],
    scopes='symbol,alias,other_names,retired',
    fields='entrezgene,symbol',
    species='human'
)
print(retry_result)

[{'query': 'C15orf26', '_id': '161502', '_score': 22.967274, 'entrezgene': '161502', 'symbol': 'CFAP161'}, {'query': 'C4orf37', '_id': '285555', '_score': 25.763496, 'entrezgene': '285555', 'symbol': 'STPG2'}, {'query': 'WDR66', '_id': '144406', '_score': 16.86136, 'entrezgene': '144406', 'symbol': 'CFAP251'}]


In [12]:
retry_mapping = {r['query']: r['entrezgene'] for r in retry_result}
gene_to_entrez.update(retry_mapping)

#drop the two entries that turned out to duplicate an already-resolved gene
del gene_to_entrez['UNC84A']
del gene_to_entrez['C4orf37']

print(len(gene_to_entrez))
print(gene_to_entrez)

21
{'LOC100049716': '100049716', 'LOC100506388': '100506388', 'LOC399886': '399886', 'ARRB1': '408', 'C15orf26': '161502', 'CYP2E1': '1571', 'FDFT1': '2222', 'LCLAT1': '253558', 'MRI1': '84245', 'NINJ2': '4815', 'OR2L13': '284521', 'PRKCZ': '5590', 'RGS14': '10636', 'RPH3AL': '9501', 'SLFN12': '55106', 'STPG2': '285555', 'SUN1': '23353', 'UGT2B15': '7366', 'UGT2B17': '7367', 'WDR66': '144406', 'ZNF718': '255403'}


For each candidate region, checks whether its gene(s) exist in the RNA-seq differential expression results at all, and if so, pulls the fold-change and p-value.

In [14]:
def lookup_rnaseq(gene_list):
    rows = []
    for gene in gene_list:
        entrez = gene_to_entrez.get(gene)
        if entrez is None:
            continue
        entrez_int = int(entrez)
        if entrez_int in rnaseq_results.index:
            row = rnaseq_results.loc[entrez_int]
            rows.append({
                'gene': gene,
                'log2FC': row['log2FoldChange'],
                'pvalue': row['pvalue'],
                'padj': row['padj']
            })
    return rows

candidates['rnaseq_hits'] = candidates['gene_list'].apply(lookup_rnaseq)
candidates[['chr', 'start_infant', 'end_infant', 'rnaseq_hits']]



,chr,start_infant,end_infant,rnaseq_hits
0,chr12,739953,740338,"[{'gene': 'LOC100049716', 'log2FC': -0.0192992..."
1,chr2,30669597,30669863,"[{'gene': 'LCLAT1', 'log2FC': 0.04067342285738..."
2,chr8,11666017,11666810,"[{'gene': 'FDFT1', 'log2FC': 0.054750934886300..."
3,chr17,180404,181288,"[{'gene': 'LOC100506388', 'log2FC': 0.81973594..."
4,chr17,33759512,33759573,"[{'gene': 'SLFN12', 'log2FC': -0.2862881151080..."
5,chr17,33759957,33759986,"[{'gene': 'SLFN12', 'log2FC': -0.2862881151080..."
6,chr15,81426347,81426669,[]
7,chr12,122356390,122356598,"[{'gene': 'WDR66', 'log2FC': 0.188162368557781..."
8,chr4,99064102,99064573,"[{'gene': 'STPG2', 'log2FC': 0.407393454854962..."
9,chr7,872130,872208,"[{'gene': 'SUN1', 'log2FC': 0.1481474660131033..."


In [15]:
flat_rows = []
for i, row in candidates.iterrows(): #Pull out p-value and check direction
    for hit in row['rnaseq_hits']:
        flat_rows.append({
            'chr': row['chr'],
            'start_infant': row['start_infant'],
            'end_infant': row['end_infant'],
            'meth_value': row['value_infant'],
            'combined_p_meth': row['combined_p'],
            'rank': row['rank'],
            'gene': hit['gene'],
            'rnaseq_log2FC': hit['log2FC'],
            'rnaseq_pvalue': hit['pvalue'],
            'rnaseq_padj': hit['padj']
        })

rnaseq_check = pd.DataFrame(flat_rows)
rnaseq_check['direction_matches_meth'] = (
    (rnaseq_check['meth_value'] > 0) == (rnaseq_check['rnaseq_log2FC'] > 0)
)

rnaseq_check.sort_values('rnaseq_pvalue')

,chr,start_infant,end_infant,meth_value,combined_p_meth,rank,gene,rnaseq_log2FC,rnaseq_pvalue,rnaseq_padj,direction_matches_meth
18,chr5,176797920,176798049,0.523620,4.564031e-06,19,RGS14,0.247525,0.000676,0.102146,True
13,chr11,74988026,74988026,-0.822031,1.745318e-06,14,ARRB1,0.286125,0.003859,0.147919,False
10,chr7,872130,872208,0.349594,1.239421e-06,10,SUN1,0.148147,0.005294,0.161300,True
6,chr17,33759512,33759573,-0.310298,7.786294e-07,5,SLFN12,-0.286288,0.039882,0.306210,True
7,chr17,33759957,33759986,-0.236441,8.809530e-07,6,SLFN12,-0.286288,0.039882,0.306210,True
1,chr12,739953,740338,0.685943,8.183002e-08,1,NINJ2,0.196632,0.114918,0.447358,True
16,chr19,13875014,13875289,-0.343781,4.213830e-06,17,MRI1,0.129206,0.163855,0.517332,False
14,chr1,2084319,2084595,0.358074,2.820209e-06,15,PRKCZ,0.088486,0.291110,0.652661,True
12,chr4,124232,124344,-0.304298,1.627315e-06,13,ZNF718,0.094537,0.321800,0.676780,False
3,chr8,11666017,11666810,-0.331600,5.194028e-07,3,FDFT1,0.054751,0.403379,0.734062,False


Now we're going to check ALL ofd them

In [16]:
all_candidates = pd.read_csv("data/all_candidates_for_rnaseq_check.csv")

all_candidates['gene_list'] = all_candidates['genes'].fillna('').apply(clean_genes)
all_candidates_genes = sorted(set(g for genes in all_candidates['gene_list'] for g in genes))

print(all_candidates.shape)
print(len(all_candidates_genes))

(4392, 14)
2854


This used a broad alias/retired search from the start, which turned out to score false matches above the correct one for genes with an exact official symbol (e.g. ACTN1 → wrong gene). Corrected in the next cell with a tiered exact-match-first approach.

In [17]:
loc_genes = [g for g in all_candidates_genes if g.startswith("LOC")]
named_genes = [g for g in all_candidates_genes if not g.startswith("LOC")]

loc_mapping = {g: g.replace("LOC", "") for g in loc_genes}

query_result = mg.querymany(
    named_genes,
    scopes='symbol,alias,other_names,retired',
    fields='entrezgene,symbol',
    species='human'
)

#Build the mapping, but track anything that failed or came back ambiguous
named_mapping = {}
failed = []
for r in query_result:
    if r.get('notfound') or 'entrezgene' not in r:
        failed.append(r['query'])
    else:
        named_mapping[r['query']] = str(r['entrezgene'])

gene_to_entrez_full = {**loc_mapping, **named_mapping}

print(f"Resolved: {len(gene_to_entrez_full)} / {len(all_candidates_genes)}")
print(f"Failed: {len(failed)}")
print(failed[:30])

610 input query terms found dup hits:	[('ABCC2', 2), ('ABCC5', 2), ('ABCG2', 2), ('ACACA', 3), ('ACAP2', 2), ('ACTN1', 2), ('ACVR1', 2), (
12 input query terms found no hit:	['C14orf184', 'CRIPAK', 'FLJ22447', 'FLJ22536', 'FLJ32810', 'FLJ35776', 'FLJ43390', 'FLJ43663', 'FLJ


Resolved: 2842 / 2854
Failed: 71
['AHCTF1P1', 'ANKRD26P1', 'ANTXRLP1', 'ATP6AP1L', 'C14orf184', 'CCL15-CCL14', 'CCL15-CCL14', 'CHEK2P2', 'CRIPAK', 'DISC1FP1', 'DLGAP1-AS1', 'DLGAP1-AS2', 'DLX6-AS1', 'FLJ22447', 'FLJ22536', 'FLJ32810', 'FLJ35776', 'FLJ43390', 'FLJ43663', 'FLJ45983', 'GRIK1-AS1', 'GUCY2EP', 'HBBP1', 'HLA-DRB6', 'HLA-L', 'HLA-L', 'HLA-L', 'HLA-L', 'HLA-L', 'HLA-L']


In [19]:
from collections import defaultdict

#exact official symbol match only
strict_result = mg.querymany(named_genes, scopes='symbol', fields='entrezgene,symbol', species='human')

strict_mapping = {}
strict_failed = []
for r in strict_result:
    if r.get('notfound') or 'entrezgene' not in r:
        strict_failed.append(r['query'])
    else:
        strict_mapping[r['query']] = str(r['entrezgene'])

print(f"Resolved on exact symbol: {len(strict_mapping)}")
print(f"Needs fallback: {len(strict_failed)}")

#try the broader alias search
fallback_result = mg.querymany(strict_failed, scopes='symbol,alias,other_names,retired',
                                fields='entrezgene,symbol', species='human')

fallback_grouped = defaultdict(list)
for r in fallback_result:
    if not r.get('notfound') and 'entrezgene' in r:
        fallback_grouped[r['query']].append(r)

fallback_mapping = {}
still_ambiguous = {}
for gene, matches in fallback_grouped.items():
    if len(matches) == 1:
        fallback_mapping[gene] = str(matches[0]['entrezgene'])
    else:
        still_ambiguous[gene] = [(m['entrezgene'], m.get('symbol'), m.get('_score')) for m in matches]

gene_to_entrez_full = {**loc_mapping, **strict_mapping, **fallback_mapping}

print(f"Resolved on fallback: {len(fallback_mapping)}")
print(f"Still ambiguous: {len(still_ambiguous)}")
print(f"Total resolved: {len(gene_to_entrez_full)} / {len(all_candidates_genes)}")
list(still_ambiguous.items())[:10]

44 input query terms found dup hits:	[('AHCTF1P1', 2), ('ANKRD26P1', 2), ('ANTXRLP1', 2), ('ATP6AP1L', 2), ('CCL15-CCL14', 3), ('CHEK2P2'
280 input query terms found no hit:	['ACCN2', 'ADSS', 'AGPAT9', 'AIM1', 'AMICA1', 'ARMC4', 'ARNTL', 'ASNA1', 'ATP5G3', 'ATP5S', 'ATPGD1'


Resolved on exact symbol: 2481
Needs fallback: 339


84 input query terms found dup hits:	[('ADSS', 2), ('AGPAT9', 2), ('AHCTF1P1', 2), ('AIM1', 4), ('ANKRD26P1', 2), ('ANTXRLP1', 2), ('ATP5
12 input query terms found no hit:	['C14orf184', 'CRIPAK', 'FLJ22447', 'FLJ22536', 'FLJ32810', 'FLJ35776', 'FLJ43390', 'FLJ43663', 'FLJ


Resolved on fallback: 264
Still genuinely ambiguous: 47
Total resolved: 2801 / 2854


[('ADSS', [('159', 'ADSS2', 17.172346), ('122622', 'ADSS1', 7.321051)]),
 ('AGPAT9', [('79888', 'LPCAT1', 19.277893), ('84803', 'GPAT3', 19.277893)]),
 ('AIM1',
  [('202', 'CRYBG1', 17.423786),
   ('9212', 'AURKB', 17.423786),
   ('51151', 'SLC45A2', 17.423786),
   ('54888', 'NSUN2', 2.8586786)]),
 ('ATP5S', [('27109', 'DMAC2L', 17.107004), ('55101', 'DMAC2', 8.443377)]),
 ('BAT2',
  [('7916', 'PRRC2A', 21.024797),
   ('23215', 'PRRC2C', 10.975806),
   ('84726', 'PRRC2B', 7.070982),
   ('360018', 'PRRC2CP1', 2.7013788)]),
 ('BRE',
  [('9577', 'BABAM2', 17.401524),
   ('100302650', 'BABAM2-AS1', 13.554705),
   ('10317', 'B3GALT5', 5.300382)]),
 ('C13orf38',
  [('728591', 'CCDC169', 24.392244),
   ('100526761', 'CCDC169-SOHLH2', 22.422152)]),
 ('C18orf21',
  [('83608', 'RMP24', 22.793926), ('100287427', 'RMP24P1', 7.3528795)]),
 ('C1orf100',
  [('200159', 'SPMIP3', 24.392244), ('127379741', 'SPMIP3P1', 7.212086)]),
 ('C20orf166',
  [('128826', 'MIR1-1HG', 25.756016), ('253868', 'CRMA', 1

This uses the loc_mapping from before the LOC-fusion-gene bug was found and fixed (see below). Superseded by the rebuild after that fix.

In [20]:
gene_to_entrez_full = {**loc_mapping, **strict_mapping, **fallback_mapping}
excluded_genes = [g for g in all_candidates_genes if g not in gene_to_entrez_full]

print(f"Final resolved: {len(gene_to_entrez_full)} / {len(all_candidates_genes)}")
print(f"Excluded: {len(excluded_genes)}")

Final resolved: 2801 / 2854
Excluded (documented, not resolved): 53


In [23]:
import re

loc_genes = [g for g in all_candidates_genes if g.startswith("LOC")]

loc_mapping = {}
loc_needs_lookup = []
for g in loc_genes:
    remainder = g.replace("LOC", "", 1)  #only strip the first occurrence, at the start
    if remainder.isdigit():
        loc_mapping[g] = remainder
    else:
        loc_needs_lookup.append(g)

print(f"Clean LOC shortcuts: {len(loc_mapping)}")
print(f"LOC-prefixed genes needing lookup: {len(loc_needs_lookup)}")
print(loc_needs_lookup[:20])

Clean LOC shortcuts: 92
LOC-prefixed genes needing real lookup: 1
['LOC100130872-SPON2']


In [24]:
loc_lookup_result = mg.querymany(loc_needs_lookup, scopes='symbol,alias,other_names,retired',
                                   fields='entrezgene,symbol', species='human')
print(loc_lookup_result)

1 input query terms found no hit:	['LOC100130872-SPON2']


[{'query': 'LOC100130872-SPON2', 'notfound': True}]


Rebuilds gene_to_entrez_full using the corrected mappings from both fixes above, then runs the same RNA-seq lookup used for the top-20 set across the full candidate list.

In [25]:
gene_to_entrez_full = {**loc_mapping, **strict_mapping, **fallback_mapping}
excluded_genes = [g for g in all_candidates_genes if g not in gene_to_entrez_full]

print(f"Final resolved: {len(gene_to_entrez_full)} / {len(all_candidates_genes)}")
print(f"Excluded: {len(excluded_genes)}")

gene_to_entrez = gene_to_entrez_full
all_candidates['rnaseq_hits'] = all_candidates['gene_list'].apply(lookup_rnaseq)

print(all_candidates['rnaseq_hits'].apply(len).sum())

Final resolved: 2800 / 2854
Excluded: 54
2999


Flattens the per-region RNA-seq hits into one row per gene-region match, keeping only direction-matched candidates, and ranks by raw RNA-seq p-value alone (see note below on why this metric needed a fix).

In [28]:
flat_rows = []
for i, row in all_candidates.iterrows():
    for hit in row['rnaseq_hits']:
        flat_rows.append({
            'chr': row['chr'],
            'start_infant': row['start_infant'],
            'end_infant': row['end_infant'],
            'meth_value': row['value_infant'],
            'combined_p_meth': row['combined_p'],
            'meth_rank': row['rank'],
            'gene': hit['gene'],
            'rnaseq_log2FC': hit['log2FC'],
            'rnaseq_pvalue': hit['pvalue'],
            'rnaseq_padj': hit['padj']
        })

full_rnaseq_check = pd.DataFrame(flat_rows)
full_rnaseq_check['direction_matches_meth'] = (
    (full_rnaseq_check['meth_value'] > 0) == (full_rnaseq_check['rnaseq_log2FC'] > 0)
)

#among genes that are BOTH direction-matched AND have a real RNA-seq signal,
#how does ISG15 (p=0.021) actually compare to the full field
concordant_both = full_rnaseq_check[full_rnaseq_check['direction_matches_meth']].copy()
concordant_both = concordant_both.sort_values('rnaseq_pvalue')
concordant_both['cross_omics_rank'] = range(1, len(concordant_both) + 1)

print(f"Total direction-matched, cross-omics-supported candidates: {len(concordant_both)}")
isg15_row = concordant_both[concordant_both['gene'] == 'ISG15']
print(isg15_row)
concordant_both.head(20)

Total direction-matched, cross-omics-supported candidates: 1584
      chr  start_infant  end_infant  meth_value  combined_p_meth  meth_rank  \
147  chr1        948625      948627   -0.223242          0.00105        215   

      gene  rnaseq_log2FC  rnaseq_pvalue  rnaseq_padj  direction_matches_meth  \
147  ISG15      -0.925921       0.020977      0.24534                    True   

     cross_omics_rank  
147               157  


,chr,start_infant,end_infant,meth_value,combined_p_meth,meth_rank,gene,rnaseq_log2FC,rnaseq_pvalue,rnaseq_padj,direction_matches_meth,cross_omics_rank
1162,chr2,242088885,242088885,0.224608,0.109656,1620,PASK,0.238016,0.000173,0.082314,True,1
2152,chr5,148728200,148728200,-0.155131,0.549701,3104,GRPEL2,-0.215133,0.000191,0.083875,True,2
726,chr1,67518911,67518911,-0.243587,0.033776,987,SLC35D1,-0.246416,0.000264,0.092478,True,3
180,chr15,42566300,42566300,-0.167395,0.001602,260,TMEM87A,-0.207867,0.000451,0.100035,True,4
1989,chr7,112090452,112090452,-0.176721,0.460066,2856,IFRD1,-0.317172,0.000476,0.100035,True,5
858,chr7,112062297,112062419,-0.231745,0.051782,1163,IFRD1,-0.317172,0.000476,0.100035,True,6
1029,chr7,112086746,112086746,-0.335758,0.078577,1406,IFRD1,-0.317172,0.000476,0.100035,True,7
481,chr5,52856476,52856570,-0.198859,0.012186,654,NDUFS4,-0.204368,0.000504,0.100035,True,8
2976,chr1,193155783,193155783,-0.156151,0.989760,4360,CDC73,-0.210330,0.000563,0.100165,True,9
1127,chr22,37678791,37678791,0.237752,0.101339,1567,CYTH4,0.500238,0.000579,0.100165,True,10


Ranking by RNA-seq p-value alone let weak methylation candidates with a decent RNA-seq hit outrank dual-evidence candidates, so combined p-values here.

In [30]:
#Combining p-value across methylation and rna-seq
from scipy.stats import chi2
import numpy as np

concordant_both['combined_p_full'] = 1 - chi2.cdf(
    -2 * (np.log(concordant_both['combined_p_meth']) + np.log(concordant_both['rnaseq_pvalue'])),
    df=4
)

concordant_both = concordant_both.sort_values('combined_p_full')
concordant_both['true_rank'] = range(1, len(concordant_both) + 1)

isg15_final = concordant_both[concordant_both['gene'] == 'ISG15']
print(isg15_final)
concordant_both.head(20)

      chr  start_infant  end_infant  meth_value  combined_p_meth  meth_rank  \
147  chr1        948625      948627   -0.223242          0.00105        215   

      gene  rnaseq_log2FC  rnaseq_pvalue  rnaseq_padj  direction_matches_meth  \
147  ISG15      -0.925921       0.020977      0.24534                    True   

     cross_omics_rank  combined_p_full  true_rank  
147               157         0.000258         49  


,chr,start_infant,end_infant,meth_value,combined_p_meth,meth_rank,gene,rnaseq_log2FC,rnaseq_pvalue,rnaseq_padj,direction_matches_meth,cross_omics_rank,combined_p_full,true_rank
19,chr5,176797920,176798049,0.523620,4.564031e-06,18,RGS14,0.247525,0.000676,0.102146,True,12,6.354264e-08,1
12,chr7,872130,872208,0.349594,1.239421e-06,10,UNC84A,0.148147,0.005294,0.161300,True,69,1.301869e-07,2
11,chr7,872130,872208,0.349594,1.239421e-06,10,SUN1,0.148147,0.005294,0.161300,True,71,1.301869e-07,3
23,chr16,1584452,1584516,0.161662,5.882337e-06,22,TMEM204,0.281529,0.001132,0.117445,True,22,1.320751e-07,4
1,chr12,739953,740338,0.685943,8.183002e-08,1,NINJ2,0.196632,0.114918,0.447358,True,449,1.832045e-07,5
22,chr16,1584452,1584516,0.161662,5.882337e-06,22,IFT140,0.264855,0.002962,0.141512,True,43,3.286524e-07,6
6,chr17,33759512,33759573,-0.310298,7.786294e-07,5,SLFN12,-0.286288,0.039882,0.306210,True,238,5.678937e-07,7
7,chr17,33759957,33759986,-0.236441,8.809530e-07,6,SLFN12,-0.286288,0.039882,0.306210,True,239,6.381855e-07,8
2,chr2,30669597,30669863,0.769272,1.138820e-07,2,LCLAT1,0.040673,0.578164,0.841544,True,1122,1.154614e-06,9
30,chr2,110969641,110970355,-0.241749,1.079838e-05,27,NCRNA00116,-0.213356,0.006840,0.175502,True,84,1.286808e-06,10


In [31]:
#does the RNA-seq analysis have ANY gene at real FDR-corrected significance,
sig_rnaseq = rnaseq_results[rnaseq_results['padj'] < 0.05]
print(f"Genes with padj < 0.05 in the full RNA-seq analysis: {len(sig_rnaseq)}")
sig_rnaseq.sort_values('padj').head(10)

Genes with padj < 0.05 in the full RNA-seq analysis: 8


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
GeneID,,,,,,
7424,4.389625,-2.083209,0.450781,-4.621331,0.000004,0.041859
389337,55.407835,0.771493,0.167666,4.601379,0.000004,0.041859
100505812,369.621565,-0.375628,0.084020,-4.470713,0.000008,0.048315
203228,175.601076,-0.341153,0.077114,-4.424008,0.000010,0.048315
6584,123.841175,0.300420,0.070403,4.267151,0.000020,0.049365
5743,17.207877,-1.211295,0.282139,-4.293260,0.000018,0.049365
57819,1787.569475,-0.285937,0.067000,-4.267742,0.000020,0.049365
10743,743.924578,0.328799,0.076109,4.320132,0.000016,0.049365


In [33]:
sig_gene_ids = [str(g) for g in sig_rnaseq.index]

id_lookup = mg.querymany(sig_gene_ids, scopes='entrezgene', fields='symbol,name', species='human')
for r in id_lookup:
    print(r.get('query'), '-', r.get('symbol'), '-', r.get('name'))

5743 - PTGS2 - prostaglandin-endoperoxide synthase 2
7424 - VEGFC - vascular endothelial growth factor C
6584 - SLC22A5 - solute carrier family 22 member 5
389337 - ARHGEF37 - Rho guanine nucleotide exchange factor 37
57819 - LSM2 - LSM2 homolog, U6 small nuclear RNA and mRNA degradation associated
203228 - C9orf72 - C9orf72-SMCR8 complex subunit
10743 - RAI1 - retinoic acid induced 1
100505812 - CARD8-AS1 - CARD8 antisense RNA 1


- Top-20 methylation candidates cross-referenced against RNA-seq: no candidate beat ISG15's RNA-seq p-value (0.021) except using a weak methylation signal (fixed by the combined-p approach above).
- Across all 4,392 candidates, 2,800/2,854 distinct genes resolved to an Entrez ID; the remainder are likely pseudogenes, etc unlikely to appear in a standard RNA-seq differential expression table.
- On the properly combined methylation+RNA-seq metric: RGS14 ranks 1st of 1,584 direction-matched, cross-omics-supported candidates and ISG15 ranks 49th
- A standalone genome-wide FDR check (independent of any methylation candidate) found 8 significant genes, most notably PTGS2 (COX-2) which has strong allergy relevance, but no corresponding methylation concordance (checked in notebook 9), consistent with it being activation-responsive rather than a resting-state marker.

Builds on notebook 9's genome-wide concordance scan.